# LeetCode 1300 — Sum of Mutated Array Closest to Target bằng Python

Link đề: https://leetcode.com/problems/sum-of-mutated-array-closest-to-target/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Sorting

Skeleton LeetCode:

```python
class Solution:
    def findBestValue(self, arr: list[int], target: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hoá bài toán thành một **hàm số** `f(v) = Σ min(a, v)` và nhận ra `f` **không giảm** theo `v`.
2. Biết **binary search trên đáp án** (search on answer): tìm `v` nhỏ nhất có `f(v) >= target`, rồi so sánh với `v − 1`.
3. Dùng **sort + prefix sum + `bisect`** để tính `f(v)` trong `O(log n)` thay vì `O(n)`.
4. Hiểu lời giải **tham lam sau khi sắp xếp**: "chia đều phần còn lại cho các phần tử chưa xử lý" và cách làm tròn đúng luật hoà.
5. Tránh các lỗi: chỉ xét một ứng viên, dùng `round()` (làm tròn kiểu ngân hàng), chọn sai cận tìm kiếm, quên đáp án có thể là `0` hoặc `max(arr)`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` và số `target`. Hãy tìm số nguyên `value` sao cho khi **thay mọi phần tử lớn hơn `value` bằng `value`**, tổng mảng mới **gần `target` nhất** (theo trị tuyệt đối của hiệu). Nếu có nhiều `value` hoà nhau, trả về `value` **nhỏ nhất**. Lưu ý `value` không nhất thiết là một phần tử của `arr`.

**Ví dụ 1:** `arr = [4,9,3], target = 10` → `3` (mảng thành `[3,3,3]`, tổng 9; `value = 4` cho tổng 11 — cũng lệch 1, nhưng 3 nhỏ hơn).

**Ví dụ 2:** `arr = [2,3,5], target = 10` → `5` (không cắt gì, tổng đúng bằng 10).

**Ví dụ 3:** `arr = [60864,25176,27249,21296,20204], target = 56803` → `11361`.

**Ràng buộc:** `1 <= arr.length <= 10^4`, `1 <= arr[i], target <= 10^5`.

In [1]:
import bisect
import random
import time
from itertools import accumulate
from typing import List

examples = [
    (([4, 9, 3], 10), 3),
    (([2, 3, 5], 10), 5),
    (([60864, 25176, 27249, 21296, 20204], 56803), 11361),
]
for (arr, target), want in examples:
    print(f"arr={arr}, target={target} -> mong đợi {want}")

arr=[4, 9, 3], target=10 -> mong đợi 3
arr=[2, 3, 5], target=10 -> mong đợi 5
arr=[60864, 25176, 27249, 21296, 20204], target=56803 -> mong đợi 11361


## 3. Kiến thức nền

### 3.1 Hàm "cắt trần" `min(a, v)` và tổng `f(v)`

Thay mọi phần tử lớn hơn `v` bằng `v` chính là lấy `min(a, v)` cho từng phần tử. Ta định nghĩa `f(v) = Σ min(a, v)`.

In [2]:
arr = [4, 9, 3]
for v in range(0, 11):
    mutated = [min(a, v) for a in arr]
    print(f"v={v:2d} -> {mutated} tổng f(v)={sum(mutated):2d}")

v= 0 -> [0, 0, 0] tổng f(v)= 0
v= 1 -> [1, 1, 1] tổng f(v)= 3
v= 2 -> [2, 2, 2] tổng f(v)= 6
v= 3 -> [3, 3, 3] tổng f(v)= 9
v= 4 -> [4, 4, 3] tổng f(v)=11
v= 5 -> [4, 5, 3] tổng f(v)=12
v= 6 -> [4, 6, 3] tổng f(v)=13
v= 7 -> [4, 7, 3] tổng f(v)=14
v= 8 -> [4, 8, 3] tổng f(v)=15
v= 9 -> [4, 9, 3] tổng f(v)=16
v=10 -> [4, 9, 3] tổng f(v)=16


### 3.2 Sắp xếp và tổng tiền tố (prefix sum)

Sau khi `sort`, `prefix[i] = arr[0] + … + arr[i-1]` (với `prefix[0] = 0`). `itertools.accumulate` tính nhanh. Tổng đoạn đầu `i` phần tử là `prefix[i]` trong `O(1)`.

In [3]:
arr = sorted([60864, 25176, 27249, 21296, 20204])
prefix = [0] + list(accumulate(arr))
print("sorted :", arr)
print("prefix :", prefix)
print("tổng 3 phần tử nhỏ nhất =", prefix[3])

sorted : [20204, 21296, 25176, 27249, 60864]
prefix : [0, 20204, 41500, 66676, 93925, 154789]
tổng 3 phần tử nhỏ nhất = 66676


### 3.3 `bisect` — tìm vị trí chèn trong mảng đã sắp xếp

- `bisect.bisect_left(a, x)`: số phần tử `< x`.
- `bisect.bisect_right(a, x)`: số phần tử `<= x`.

Với `f(v)`: các phần tử `<= v` giữ nguyên, các phần tử `> v` thành `v`. Gọi `i = bisect_right(arr, v)` ⇒ `f(v) = prefix[i] + v·(n − i)`.

In [4]:
arr = [3, 4, 9]
prefix = [0] + list(accumulate(arr))
for v in [0, 3, 5, 9, 12]:
    i = bisect.bisect_right(arr, v)
    print(f"v={v:2d}: bisect_right={i}, f(v)={prefix[i] + v * (len(arr) - i)}, kiểm tra={sum(min(a, v) for a in arr)}")

v= 0: bisect_right=0, f(v)=0, kiểm tra=0
v= 3: bisect_right=1, f(v)=9, kiểm tra=9
v= 5: bisect_right=2, f(v)=12, kiểm tra=12
v= 9: bisect_right=3, f(v)=16, kiểm tra=16
v=12: bisect_right=3, f(v)=16, kiểm tra=16


### 3.4 Binary search "tìm vị trí đầu tiên thoả điều kiện"

Khi điều kiện `P(v)` có dạng `False … False True … True` (đơn điệu), ta tìm `v` nhỏ nhất có `P(v) = True` bằng khung:

```python
lo, hi = L, R          # đáp án nằm trong [L, R]
while lo < hi:
    mid = (lo + hi) // 2
    if P(mid): hi = mid
    else:      lo = mid + 1
return lo
```

In [5]:
def first_true(lo, hi, P):
    while lo < hi:
        mid = (lo + hi) // 2
        if P(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print("số nhỏ nhất có x*x >= 50 trong [0, 100]:", first_true(0, 100, lambda x: x * x >= 50))

số nhỏ nhất có x*x >= 50 trong [0, 100]: 8


### 3.5 Chia nguyên, so sánh hai ứng viên và luật hoà

`q = rem // cnt` là phần nguyên. Giá trị tốt nhất là `q` hoặc `q + 1`. So sánh `|q·cnt − rem|` với `|(q+1)·cnt − rem|`; nếu bằng nhau chọn `q` (nhỏ hơn). **Đừng dùng `round()`**: Python làm tròn "về số chẵn" (`round(3.5) = 4`, `round(2.5) = 2`).

In [6]:
print("round(2.5) =", round(2.5), "| round(3.5) =", round(3.5), "-> không phải luôn làm tròn xuống khi hoà!")
rem, cnt = 7, 2
q = rem // cnt
print("q =", q, "| lệch của q:", abs(q * cnt - rem), "| lệch của q+1:", abs((q + 1) * cnt - rem))

round(2.5) = 2 | round(3.5) = 4 -> không phải luôn làm tròn xuống khi hoà!
q = 3 | lệch của q: 1 | lệch của q+1: 1


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**: `arr = [4, 9, 3]`, `target = 10`.

| v | mảng sau khi cắt | f(v) | \|f(v) − 10\| |
|---|---|---|---|
| 0 | [0,0,0] | 0 | 10 |
| 1 | [1,1,1] | 3 | 7 |
| 2 | [2,2,2] | 6 | 4 |
| 3 | [3,3,3] | 9 | **1** |
| 4 | [4,4,3] | 11 | **1** |
| 5 | [4,5,3] | 12 | 2 |
| 9 | [4,9,3] | 16 | 6 |
| ≥ 9 | không đổi | 16 | 6 |

Quan sát:
1. `f(v)` **tăng dần (không giảm)** khi `v` tăng, và **dừng lại** khi `v >= max(arr)`.
2. `|f(v) − target|` giảm rồi tăng (hình chữ V/"thung lũng").
3. Hai ứng viên tốt nhất nằm **sát hai bên** điểm mà `f(v)` vượt qua `target`: `v = 3` (dưới) và `v = 4` (trên). Hoà ⇒ chọn `3`.

In [7]:
arr, target = [4, 9, 3], 10
for v in range(0, 11):
    f = sum(min(a, v) for a in arr)
    print(f"v={v:2d} f={f:2d} lệch={abs(f - target):2d} " + "#" * abs(f - target))

v= 0 f= 0 lệch=10 ##########
v= 1 f= 3 lệch= 7 #######
v= 2 f= 6 lệch= 4 ####
v= 3 f= 9 lệch= 1 #
v= 4 f=11 lệch= 1 #
v= 5 f=12 lệch= 2 ##
v= 6 f=13 lệch= 3 ###
v= 7 f=14 lệch= 4 ####
v= 8 f=15 lệch= 5 #####
v= 9 f=16 lệch= 6 ######
v=10 f=16 lệch= 6 ######


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** đáp án nằm trong `[0, max(arr)]` (vượt quá `max(arr)` thì `f` không đổi, mà hoà phải chọn nhỏ nhất). Thử mọi `v` trong đoạn đó, tính `f(v)` trong `O(n)`, giữ `v` có độ lệch nhỏ nhất; duyệt tăng dần và chỉ cập nhật khi **lệch nhỏ hơn hẳn** ⇒ tự động chọn `v` nhỏ nhất khi hoà.

Vì sao xét cả `v = 0`? Ví dụ `arr = [5,5,5], target = 1`: `v = 0` cho tổng 0 (lệch 1), `v = 1` cho tổng 3 (lệch 2) ⇒ đáp án `0`.

In [8]:
def best_value_brute(arr, target):
    best_v, best_diff = 0, float("inf")
    for v in range(0, max(arr) + 1):
        diff = abs(sum(min(a, v) for a in arr) - target)
        if diff < best_diff:            # chỉ "<" -> giữ v nhỏ nhất khi hoà
            best_v, best_diff = v, diff
    return best_v

for (arr, target), want in examples:
    t0 = time.perf_counter()
    got = best_value_brute(arr, target)
    print(arr, target, "->", got, "| mong đợi", want, f"({time.perf_counter() - t0:.3f}s)")
print("[5,5,5], target=1 ->", best_value_brute([5, 5, 5], 1))

[4, 9, 3] 10 -> 3 | mong đợi 3 (0.000s)
[2, 3, 5] 10 -> 5 | mong đợi 5 (0.000s)
[60864, 25176, 27249, 21296, 20204] 56803 -> 11361 | mong đợi 11361 (0.032s)
[5,5,5], target=1 -> 0


**Độ phức tạp:** gọi `n = len(arr)`, `M = max(arr)`.
- Time `O(n · M)`: tới `10^4 · 10^5 = 10^9` phép tính ⇒ **quá chậm** cho LeetCode.
- Space `O(1)`.

**Nút thắt:** (1) thử từng `v` một trong khi `f` đơn điệu — có thể nhảy nhị phân; (2) mỗi lần tính `f(v)` tốn `O(n)` — có thể dùng prefix sum.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — tính đơn điệu:** khi `v` tăng 1, mỗi `min(a, v)` tăng 0 hoặc 1 ⇒ `f(v)` **không giảm**. Vậy tồn tại ranh giới: `f(v) < target` với `v` nhỏ, `f(v) >= target` từ một điểm `v*` trở đi (hoặc không bao giờ nếu `sum(arr) < target`).

**Insight 2 — chỉ hai ứng viên:** vì `f` không giảm:
- Mọi `v < v* − 1` có `f(v) <= f(v* − 1) < target` ⇒ lệch không nhỏ hơn của `v* − 1`.
- Mọi `v > v*` có `f(v) >= f(v*) >= target` ⇒ lệch không nhỏ hơn của `v*`.
- ⇒ Đáp án là `v* − 1` hoặc `v*`. **Nếu** lệch bằng nhau **thì** chọn `v* − 1`.

**Insight 3 — tính `f` nhanh:** sắp xếp `arr`, `f(v) = prefix[i] + v·(n − i)` với `i = bisect_right(arr, v)`.

**Insight 4 — tham lam:** duyệt mảng đã sắp xếp; ở bước `i`, giả sử mọi phần tử từ `i` trở đi bị cắt về cùng giá trị `v`, cần `v·(n − i) ≈ target − prefix[i]`. Nếu `arr[i]·(n − i) >= target − prefix[i]` thì `v` nằm trong đoạn này ⇒ lấy `v ≈ rem / cnt`.

In [9]:
# Minh hoạ: f(v) không giảm, và ranh giới v*
arr, target = [4, 9, 3], 10
fs = [sum(min(a, v) for a in arr) for v in range(0, 12)]
print("f(v), v=0..11:", fs)
assert all(fs[i] <= fs[i + 1] for i in range(len(fs) - 1))
v_star = next(v for v, f in enumerate(fs) if f >= target)
print("v* =", v_star, "| ứng viên:", v_star - 1, "(f=%d)" % fs[v_star - 1], "và", v_star, "(f=%d)" % fs[v_star])

f(v), v=0..11: [0, 3, 6, 9, 11, 12, 13, 14, 15, 16, 16, 16]
v* = 4 | ứng viên: 3 (f=9) và 4 (f=11)


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Binary search trên `value`, tính `f` trong `O(n)`

Chưa cần sắp xếp: tìm `v*` = `v` nhỏ nhất trong `[0, max(arr)]` có `f(v) >= target`. Nếu `sum(arr) <= target` thì đáp án luôn là `max(arr)` (không cắt gì là tốt nhất, và đó là `v` nhỏ nhất cho tổng lớn nhất).

Các bước:
1. `M = max(arr)`; nếu `sum(arr) <= target` ⇒ trả `M`.
2. Binary search `first_true` trên `[0, M]` với `P(v) = f(v) >= target` ⇒ `v*`. (`P(M)` chắc chắn đúng.)
3. So `v* − 1` và `v*` (nếu `v* = 0` thì chỉ có một ứng viên).

In [10]:
def best_value_bs(arr, target):
    def f(v):
        return sum(min(a, v) for a in arr)
    M = max(arr)
    if sum(arr) <= target:
        return M
    lo, hi = 0, M
    while lo < hi:
        mid = (lo + hi) // 2
        if f(mid) >= target:
            hi = mid
        else:
            lo = mid + 1
    v = lo                                   # v* nhỏ nhất có f(v) >= target
    if v > 0 and target - f(v - 1) <= f(v) - target:
        return v - 1                         # hoà -> chọn nhỏ hơn
    return v

for (arr, target), want in examples:
    print(arr, target, "->", best_value_bs(arr, target), "| mong đợi", want)

[4, 9, 3] 10 -> 3 | mong đợi 3
[2, 3, 5] 10 -> 5 | mong đợi 5
[60864, 25176, 27249, 21296, 20204] 56803 -> 11361 | mong đợi 11361


Code nộp LeetCode (cách 2):

In [11]:
class SolutionBS:
    def findBestValue(self, arr: list[int], target: int) -> int:
        def f(v: int) -> int:
            return sum(min(a, v) for a in arr)
        M = max(arr)
        if sum(arr) <= target:
            return M
        lo, hi = 0, M
        while lo < hi:
            mid = (lo + hi) // 2
            if f(mid) >= target:
                hi = mid
            else:
                lo = mid + 1
        if lo > 0 and target - f(lo - 1) <= f(lo) - target:
            return lo - 1
        return lo

print(SolutionBS().findBestValue([4, 9, 3], 10))

3


**Độ phức tạp cách 2:** Time `O(n · log M)` ≈ `10^4 · 17` — rất nhanh. Space `O(1)`.

### 7.2 Cách 3 — Sort + prefix sum + binary search (`f` trong `O(log n)`)

Sắp xếp một lần `O(n log n)`, dựng `prefix`. Mỗi lần tính `f(v)` chỉ cần `bisect_right` + một phép nhân. Phần binary search giờ chỉ `O(log M · log n)` — có ích khi phải trả lời **nhiều** `target` trên cùng một mảng.

In [12]:
def best_value_prefix(arr, target):
    a = sorted(arr)
    n = len(a)
    prefix = [0] + list(accumulate(a))
    def f(v):
        i = bisect.bisect_right(a, v)
        return prefix[i] + v * (n - i)
    if prefix[n] <= target:
        return a[-1]
    lo, hi = 0, a[-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if f(mid) >= target:
            hi = mid
        else:
            lo = mid + 1
    if lo > 0 and target - f(lo - 1) <= f(lo) - target:
        return lo - 1
    return lo

for (arr, target), want in examples:
    print(arr, target, "->", best_value_prefix(arr, target), "| mong đợi", want)

[4, 9, 3] 10 -> 3 | mong đợi 3
[2, 3, 5] 10 -> 5 | mong đợi 5
[60864, 25176, 27249, 21296, 20204] 56803 -> 11361 | mong đợi 11361


Code nộp LeetCode (cách 3):

In [13]:
class SolutionPrefix:
    def findBestValue(self, arr: list[int], target: int) -> int:
        a = sorted(arr)
        n = len(a)
        prefix = [0] * (n + 1)
        for i, x in enumerate(a):
            prefix[i + 1] = prefix[i] + x

        def f(v: int) -> int:
            i = bisect.bisect_right(a, v)
            return prefix[i] + v * (n - i)

        if prefix[n] <= target:
            return a[-1]
        lo, hi = 0, a[-1]
        while lo < hi:
            mid = (lo + hi) // 2
            if f(mid) >= target:
                hi = mid
            else:
                lo = mid + 1
        if lo > 0 and target - f(lo - 1) <= f(lo) - target:
            return lo - 1
        return lo

print(SolutionPrefix().findBestValue([60864, 25176, 27249, 21296, 20204], 56803))

11361


**Độ phức tạp cách 3:** Time `O(n log n + log M · log n)`, Space `O(n)` cho mảng sắp xếp + prefix.

### 7.3 Cách 4 — Sort + tham lam "chia đều phần còn lại"

Duyệt mảng đã sắp xếp tăng dần, giữ `rem` = phần `target` còn thiếu sau khi đã giữ nguyên các phần tử nhỏ, `cnt = n − i` = số phần tử chưa xử lý.

- **Nếu** `a[i] · cnt >= rem`: cắt hết `cnt` phần tử còn lại về cùng giá trị `v` là đủ, và `v` nằm trong đoạn `[a[i-1], a[i]]` nơi `f` tuyến tính với hệ số `cnt`. Lấy `q = rem // cnt`; so `q` với `q + 1` (luật hoà: chọn `q` khi `rem − q·cnt <= (q+1)·cnt − rem`). Trả về.
- **Ngược lại:** `a[i]` giữ nguyên, `rem −= a[i]`, sang phần tử tiếp theo.
- Duyệt hết mà chưa trả về ⇒ `sum(arr) < target` ⇒ trả `a[-1]`.

In [14]:
def best_value_greedy(arr, target):
    a = sorted(arr)
    n = len(a)
    rem = target
    for i, x in enumerate(a):
        cnt = n - i
        if x * cnt >= rem:
            q = rem // cnt
            if rem - q * cnt <= (q + 1) * cnt - rem:
                return q
            return q + 1
        rem -= x
    return a[-1]

for (arr, target), want in examples:
    print(arr, target, "->", best_value_greedy(arr, target), "| mong đợi", want)

[4, 9, 3] 10 -> 3 | mong đợi 3
[2, 3, 5] 10 -> 5 | mong đợi 5
[60864, 25176, 27249, 21296, 20204] 56803 -> 11361 | mong đợi 11361


Code nộp LeetCode (cách 4):

In [15]:
class SolutionGreedy:
    def findBestValue(self, arr: list[int], target: int) -> int:
        a = sorted(arr)
        n = len(a)
        rem = target
        for i, x in enumerate(a):
            cnt = n - i
            if x * cnt >= rem:
                q = rem // cnt
                return q if rem - q * cnt <= (q + 1) * cnt - rem else q + 1
            rem -= x
        return a[-1]

print(SolutionGreedy().findBestValue([2, 3, 5], 10))

5


**Độ phức tạp cách 4:** Time `O(n log n)` (bị chi phối bởi sort; vòng duyệt `O(n)`), Space `O(n)` cho bản sắp xếp (hoặc `O(1)` thêm nếu sort tại chỗ).

**Vì sao `q + 1 <= a[i]`?** Nếu `rem / cnt` nguyên thì `q = rem / cnt <= a[i]` và lệch 0 ⇒ trả `q`. Nếu không nguyên thì `q < rem / cnt <= a[i]` ⇒ `q + 1 <= a[i]`. Và `q >= a[i−1]` vì bước trước thất bại: `a[i−1]·(cnt+1) < rem + a[i−1]` ⇒ `a[i−1] < rem / cnt`. Vậy cả hai ứng viên đều nằm trong đoạn tuyến tính — công thức `f(v) = prefix[i] + v·cnt` đúng.

In [16]:
rng = random.Random(3)
arr_big = [rng.randint(1, 10**5) for _ in range(10**4)]
tg = 10**5
for name, fn in [("bs O(n log M)", best_value_bs), ("prefix", best_value_prefix), ("greedy", best_value_greedy)]:
    t0 = time.perf_counter()
    r = fn(arr_big, tg)
    print(f"{name:14s} -> {r:6d} ({time.perf_counter() - t0:.4f}s)")

bs O(n log M)  ->     10 (0.0083s)
prefix         ->     10 (0.0018s)
greedy         ->     10 (0.0014s)


## 8. TRACE từng bước

### 8.1 Trace cách 4 (tham lam) trên Ví dụ 3

`arr` sắp xếp: `[20204, 21296, 25176, 27249, 60864]`, `target = 56803`, `n = 5`.

| i | a[i] | cnt | rem | a[i]·cnt | a[i]·cnt >= rem? | hành động |
|---|---|---|---|---|---|---|
| 0 | 20204 | 5 | 56803 | 101020 | ✔ | q = 56803 // 5 = 11360; lệch q: 3, lệch q+1: 2 ⇒ trả **11361** |

Ngay bước đầu đã dừng: mọi phần tử đều bị cắt về khoảng 11361 (vì kể cả phần tử nhỏ nhất cũng lớn hơn).

In [17]:
def trace_greedy(arr, target):
    a = sorted(arr)
    n = len(a)
    rem = target
    print("sorted:", a, "| target:", target)
    for i, x in enumerate(a):
        cnt = n - i
        print(f"i={i} a[i]={x} cnt={cnt} rem={rem} a[i]*cnt={x * cnt}", end=" ")
        if x * cnt >= rem:
            q = rem // cnt
            dq, dq1 = rem - q * cnt, (q + 1) * cnt - rem
            ans = q if dq <= dq1 else q + 1
            print(f"-> dừng: q={q}, lệch(q)={dq}, lệch(q+1)={dq1} => {ans}")
            return ans
        print(f"-> giữ nguyên {x}, rem = {rem} - {x} = {rem - x}")
        rem -= x
    print("-> duyệt hết, tổng < target => trả max =", a[-1])
    return a[-1]

trace_greedy([60864, 25176, 27249, 21296, 20204], 56803)

sorted: [20204, 21296, 25176, 27249, 60864] | target: 56803
i=0 a[i]=20204 cnt=5 rem=56803 a[i]*cnt=101020 -> dừng: q=11360, lệch(q)=3, lệch(q+1)=2 => 11361


11361

In [18]:
trace_greedy([2, 3, 5], 9)
print()
trace_greedy([4, 9, 3], 10)

sorted: [2, 3, 5] | target: 9
i=0 a[i]=2 cnt=3 rem=9 a[i]*cnt=6 -> giữ nguyên 2, rem = 9 - 2 = 7
i=1 a[i]=3 cnt=2 rem=7 a[i]*cnt=6 -> giữ nguyên 3, rem = 7 - 3 = 4
i=2 a[i]=5 cnt=1 rem=4 a[i]*cnt=5 -> dừng: q=4, lệch(q)=0, lệch(q+1)=1 => 4

sorted: [3, 4, 9] | target: 10
i=0 a[i]=3 cnt=3 rem=10 a[i]*cnt=9 -> giữ nguyên 3, rem = 10 - 3 = 7
i=1 a[i]=4 cnt=2 rem=7 a[i]*cnt=8 -> dừng: q=3, lệch(q)=1, lệch(q+1)=1 => 3


3

### 8.2 Trace cách 2 (binary search) trên Ví dụ 1

`arr = [4, 9, 3]`, `target = 10`, `M = 9`, `sum = 16 > 10`.

| lo | hi | mid | f(mid) | f(mid) >= 10? | cập nhật |
|---|---|---|---|---|---|
| 0 | 9 | 4 | 11 | ✔ | hi = 4 |
| 0 | 4 | 2 | 6 | ✘ | lo = 3 |
| 3 | 4 | 3 | 9 | ✘ | lo = 4 |

Dừng `v* = 4`. So `v* − 1 = 3`: lệch `10 − 9 = 1`, `v* = 4`: lệch `11 − 10 = 1` ⇒ hoà ⇒ **3**.

In [19]:
def trace_bs(arr, target):
    f = lambda v: sum(min(a, v) for a in arr)
    M = max(arr)
    if sum(arr) <= target:
        print("sum <= target -> trả max:", M)
        return M
    lo, hi = 0, M
    while lo < hi:
        mid = (lo + hi) // 2
        ok = f(mid) >= target
        print(f"lo={lo} hi={hi} mid={mid} f(mid)={f(mid)} {'>=' if ok else '<'} {target}", end=" ")
        if ok:
            hi = mid
        else:
            lo = mid + 1
        print(f"-> lo={lo} hi={hi}")
    v = lo
    if v > 0:
        print(f"v*={v}: lệch(v*-1)={target - f(v - 1)}, lệch(v*)={f(v) - target}")
    ans = v - 1 if v > 0 and target - f(v - 1) <= f(v) - target else v
    print("=> đáp án", ans)
    return ans

trace_bs([4, 9, 3], 10)

lo=0 hi=9 mid=4 f(mid)=11 >= 10 -> lo=0 hi=4
lo=0 hi=4 mid=2 f(mid)=6 < 10 -> lo=3 hi=4
lo=3 hi=4 mid=3 f(mid)=9 < 10 -> lo=4 hi=4
v*=4: lệch(v*-1)=1, lệch(v*)=1
=> đáp án 3


3

## 9. Code nộp LeetCode

Bản chính thức: **sort + tham lam** — ngắn, `O(n log n)`, không cần binary search.

In [20]:
class Solution:
    def findBestValue(self, arr: list[int], target: int) -> int:
        a = sorted(arr)
        n = len(a)
        rem = target                      # phần target còn thiếu
        for i, x in enumerate(a):
            cnt = n - i                   # số phần tử sẽ bị cắt về cùng giá trị
            if x * cnt >= rem:
                q = rem // cnt
                # chọn q hoặc q + 1; hoà -> chọn nhỏ hơn
                return q if rem - q * cnt <= (q + 1) * cnt - rem else q + 1
            rem -= x                      # x giữ nguyên
        return a[-1]                      # tổng toàn mảng vẫn < target

sol = Solution()
for (arr, target), want in examples:
    got = sol.findBestValue(arr, target)
    print(arr, target, "->", got, "✔" if got == want else "✘")

[4, 9, 3] 10 -> 3 ✔
[2, 3, 5] 10 -> 5 ✔
[60864, 25176, 27249, 21296, 20204] 56803 -> 11361 ✔


## 10. Vì sao thuật toán đúng

**Bổ đề (đơn điệu):** `f(v) = Σ min(a, v)` không giảm theo `v`, và `f(v) = sum(arr)` với mọi `v >= max(arr)`.

**Binary search (cách 2, 3):**
- Nếu `sum(arr) <= target`: mọi `v` cho `f(v) <= sum(arr) <= target`, lệch `target − f(v)` nhỏ nhất khi `f(v)` lớn nhất = `sum(arr)`, đạt lần đầu tại `v = max(arr)` (với `v < max(arr)` thì `f(v) < sum(arr)`). ⇒ trả `max(arr)` đúng.
- Ngược lại, `v*` tồn tại trong `[0, M]`. Như mục 6: mọi `v < v*−1` không tốt hơn `v*−1`; mọi `v > v*` không tốt hơn `v*`. So hai ứng viên, ưu tiên nhỏ hơn khi hoà. Còn phải chắc chắn không có `v < v*−1` nào **hoà** với `v*−1` (vì khi đó đề yêu cầu trả `v` nhỏ hơn nữa) — xem lưu ý dưới.

**Lưu ý luật hoà khi `f` có "bậc thang phẳng":** `f(v)` tăng **thực sự** (ít nhất 1) mỗi khi `v < max(arr)` vì luôn có ít nhất phần tử lớn nhất bị cắt ⇒ trên `[0, max(arr)]`, `f` **tăng chặt**. Do đó không có `v` khác cùng giá trị `f` với `v*−1`, luật hoà chỉ cần xét giữa `v*−1` và `v*`.

**Tham lam (cách 4):** ở bước `i` đầu tiên có `a[i]·cnt >= rem`, ta đã chứng minh (mục 7.3) cả `q` và `q+1` nằm trong đoạn `[a[i−1], a[i]]` nơi `f(v) = prefix[i] + v·cnt`. Khi đó `f(v) − target = v·cnt − rem`, nên `v* ∈ {q, q+1}` và hai ứng viên của binary search chính là `q`, `q + 1` (hoặc chỉ `q` khi chia hết). ⇒ cùng kết quả với cách 2.

## 11. Phân tích Time / Space

Gọi `n = len(arr)` (≤ `10^4`), `M = max(arr)` (≤ `10^5`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n·M)` ≈ `10^9` | `O(1)` | chỉ dùng kiểm chứng |
| 2. Binary search, `f` `O(n)` | `O(n log M)` | `O(1)` | không cần sort |
| 3. Sort + prefix + bisect | `O(n log n + log M · log n)` | `O(n)` | tốt khi nhiều truy vấn `target` |
| 4. Sort + tham lam | `O(n log n)` | `O(n)` | bản nộp chính |

Không có khác biệt average/worst đáng kể: mọi cách đều tất định, chi phí không phụ thuộc phân bố dữ liệu (trừ cách 4 có thể dừng sớm, nhưng `sort` vẫn `O(n log n)`).

## 12. Bộ test

Ví dụ đề, edge case (1 phần tử, đáp án `0`, `sum < target`, `sum == target`, hoà, mọi phần tử bằng nhau), rồi đối chiếu tất cả các cách với brute force trên input ngẫu nhiên (seed cố định, giá trị nhỏ để brute force chạy nhanh).

In [21]:
ALL = [best_value_brute, best_value_bs, best_value_prefix, best_value_greedy,
       Solution().findBestValue, SolutionBS().findBestValue, SolutionPrefix().findBestValue, SolutionGreedy().findBestValue]
tests = [
    (([4, 9, 3], 10), 3),
    (([2, 3, 5], 10), 5),
    (([60864, 25176, 27249, 21296, 20204], 56803), 11361),
    (([5], 3), 3),                 # 1 phần tử, cắt về đúng target
    (([5], 100), 5),               # tổng < target -> max
    (([5, 5, 5], 1), 0),           # đáp án 0
    (([10, 10], 7), 3),            # hoà 3 (6) và 4 (8) -> 3
    (([1, 2, 3], 6), 3),           # sum == target
    (([7, 7, 7, 7], 20), 5),       # mọi phần tử bằng nhau
    (([1, 1, 100], 52), 50),
]
for (arr, target), want in tests:
    for f in ALL:
        got = f(list(arr), target)
        assert got == want, (f, arr, target, got, want)
print("Tất cả", len(tests), "test cố định qua cho", len(ALL), "hàm")

Tất cả 10 test cố định qua cho 8 hàm


In [22]:
rng = random.Random(1300)
cases = 0
for _ in range(2000):
    n = rng.randint(1, 8)
    arr = [rng.randint(1, 30) for _ in range(n)]
    target = rng.randint(1, 120)
    want = best_value_brute(arr, target)
    for f in ALL[1:]:
        assert f(list(arr), target) == want, (f, arr, target)
    cases += 1
print("Đối chiếu ngẫu nhiên:", cases, "bộ test khớp brute force")

Đối chiếu ngẫu nhiên: 2000 bộ test khớp brute force


In [23]:
# Kiểm tra ở kích thước tối đa: các cách nhanh phải khớp nhau
rng = random.Random(77)
for _ in range(5):
    arr = [rng.randint(1, 10**5) for _ in range(10**4)]
    target = rng.randint(1, 10**5)
    r = {f(list(arr), target) for f in [best_value_bs, best_value_prefix, best_value_greedy]}
    assert len(r) == 1, r
t0 = time.perf_counter()
Solution().findBestValue(arr, target)
print("n=1e4 khớp nhau; Solution chạy", f"{time.perf_counter() - t0:.4f}s")

n=1e4 khớp nhau; Solution chạy 0.0012s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ trả về `v*` (giá trị nhỏ nhất có `f(v) >= target`)

Quên rằng `v* − 1` (tổng hơi dưới target) có thể gần hơn hoặc hoà.

In [24]:
def wrong_only_vstar(arr, target):
    f = lambda v: sum(min(a, v) for a in arr)
    if sum(arr) <= target:
        return max(arr)
    lo, hi = 0, max(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if f(mid) >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo                                   # SAI: không so với lo - 1

print("[4,9,3],10 -> sai:", wrong_only_vstar([4, 9, 3], 10), "| đúng:", Solution().findBestValue([4, 9, 3], 10))
print("[10,10,10],10 -> sai:", wrong_only_vstar([10, 10, 10], 10), "| đúng:", Solution().findBestValue([10, 10, 10], 10), "(f(3)=9 lệch 1, f(4)=12 lệch 2)")

[4,9,3],10 -> sai: 4 | đúng: 3
[10,10,10],10 -> sai: 4 | đúng: 3 (f(3)=9 lệch 1, f(4)=12 lệch 2)


**Vì sao sai:** `f(4) = 11` lệch 1, `f(3) = 9` cũng lệch 1 ⇒ phải chọn 3; với `[10,10,10]` thì `v* − 1 = 3` còn gần hơn hẳn. **Sửa:** sau binary search luôn so `v* − 1` và `v*` (xem `SolutionBS`).

### Lỗi 2 — Dùng `round(rem / cnt)` trong lời giải tham lam

`round` của Python làm tròn **về số chẵn** khi đúng `.5`, trong khi đề yêu cầu hoà thì lấy **nhỏ hơn**.

In [25]:
def wrong_round(arr, target):
    a = sorted(arr)
    n, rem = len(a), target
    for i, x in enumerate(a):
        cnt = n - i
        if x * cnt >= rem:
            return round(rem / cnt)            # SAI ở trường hợp .5
        rem -= x
    return a[-1]

print("[10,10],7  -> sai:", wrong_round([10, 10], 7), "| đúng:", Solution().findBestValue([10, 10], 7))
print("[10,10],5  ->     ", wrong_round([10, 10], 5), "| đúng:", Solution().findBestValue([10, 10], 5), "(may mắn đúng vì 2.5 -> 2)")

[10,10],7  -> sai: 4 | đúng: 3
[10,10],5  ->      2 | đúng: 2 (may mắn đúng vì 2.5 -> 2)


**Vì sao sai:** `7 / 2 = 3.5`, `round(3.5) = 4` (số chẵn) nhưng `v = 3` (tổng 6) và `v = 4` (tổng 8) hoà ⇒ phải trả 3. Cộng thêm sai số dấu phẩy động với số lớn. **Sửa:** dùng số nguyên `q = rem // cnt` và so trực tiếp hai độ lệch.

### Lỗi 3 — Đặt cận trên tìm kiếm là `target` thay vì `max(arr)`, và không xử lý `sum(arr) < target`

In [26]:
def wrong_bound(arr, target):
    f = lambda v: sum(min(a, v) for a in arr)
    lo, hi = 0, target                          # SAI: cận trên nên là max(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if f(mid) >= target:
            hi = mid
        else:
            lo = mid + 1
    if lo > 0 and target - f(lo - 1) <= f(lo) - target:
        return lo - 1
    return lo

print("[2,3,5],100 -> sai:", wrong_bound([2, 3, 5], 100), "| đúng:", Solution().findBestValue([2, 3, 5], 100))

[2,3,5],100 -> sai: 100 | đúng: 5


**Vì sao sai:** khi `sum(arr) < target`, không `v` nào có `f(v) >= target`, binary search trôi tới cận trên `100`; mọi `v >= 5` đều hoà nhưng đề yêu cầu `v` nhỏ nhất là `5`. **Sửa:** kiểm tra `sum(arr) <= target` thì trả `max(arr)`, và tìm trong `[0, max(arr)]`.

### Lỗi 4 — Bắt đầu tìm từ `1`, bỏ sót đáp án `0`

In [27]:
def wrong_from_one(arr, target):
    best_v, best_diff = 1, float("inf")
    for v in range(1, max(arr) + 1):            # SAI: bỏ qua v = 0
        d = abs(sum(min(a, v) for a in arr) - target)
        if d < best_diff:
            best_v, best_diff = v, d
    return best_v

print("[5,5,5],1 -> sai:", wrong_from_one([5, 5, 5], 1), "| đúng:", Solution().findBestValue([5, 5, 5], 1))

[5,5,5],1 -> sai: 1 | đúng: 0


**Vì sao sai:** `v = 0` cho tổng 0 (lệch 1) tốt hơn `v = 1` (tổng 3, lệch 2). Khi `n > target` điều này xảy ra. **Sửa:** miền tìm kiếm là `[0, max(arr)]` — lời giải tham lam tự xử lý vì `q = rem // cnt` có thể bằng 0.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tìm một giá trị nguyên sao cho một đại lượng gần/đạt target"** + đại lượng **đơn điệu** theo giá trị ⇒ **binary search on answer**.
- Sau binary search kiểu "gần nhất", **luôn kiểm tra hai ứng viên kề ranh giới**.
- **"Thay mọi phần tử > v bằng v"** (cap / clamp) ⇒ sort + prefix sum: `f(v) = prefix[i] + v·(n−i)`.
- Gặp đề "chia đều phần còn lại" ⇒ sort tăng dần, xử lý phần tử nhỏ trước (chúng không bị cắt).
- **Luật hoà "chọn nhỏ nhất"** ⇒ duyệt tăng dần với so sánh `<` chặt, hoặc so `<=` cho ứng viên nhỏ.
- Tránh số thực: so sánh bằng phép nhân số nguyên.

## 15. Biến thể của bài

1. **Nhiều truy vấn `target`** trên cùng `arr`: sort + prefix một lần, mỗi truy vấn binary search `O(log M · log n)` (cách 3).
2. **"Nâng sàn"**: thay mọi phần tử **nhỏ hơn** `v` bằng `v` — `g(v) = Σ max(a, v)` cũng không giảm (bài tập 5).
3. **Tổng ≤ target, `v` lớn nhất** (không cần "gần nhất"): chỉ cần binary search một phía.
4. **Ternary search** trên `|f(v) − target|` (hàm thung lũng) — gợi ý của đề; cần cẩn thận vùng phẳng khi `v >= max(arr)`.
5. **LeetCode 1011 / 875 / 410** — cùng mẫu binary search trên đáp án với hàm kiểm tra đơn điệu.
6. **Phân bổ ngân sách (budget cap)**: chọn mức trần trợ cấp sao cho tổng chi gần ngân sách nhất — đúng bài này trong ngữ cảnh thực tế.

In [28]:
# Biến thể 1: nhiều truy vấn target trên cùng mảng
class CapQuery:
    def __init__(self, arr):
        self.a = sorted(arr)
        self.n = len(self.a)
        self.p = [0] + list(accumulate(self.a))
    def f(self, v):
        i = bisect.bisect_right(self.a, v)
        return self.p[i] + v * (self.n - i)
    def best(self, target):
        if self.p[-1] <= target:
            return self.a[-1]
        lo, hi = 0, self.a[-1]
        while lo < hi:
            mid = (lo + hi) // 2
            if self.f(mid) >= target:
                hi = mid
            else:
                lo = mid + 1
        return lo - 1 if lo > 0 and target - self.f(lo - 1) <= self.f(lo) - target else lo

cq = CapQuery([4, 9, 3])
print({t: cq.best(t) for t in [1, 5, 10, 12, 16, 30]})
assert all(cq.best(t) == best_value_brute([4, 9, 3], t) for t in range(1, 40))

{1: 0, 5: 2, 10: 3, 12: 5, 16: 9, 30: 9}


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Thử mọi `v ∈ [0, max]`, tính `f` `O(n)` | `O(n·M)` | `O(1)` | Kiểm chứng trên input nhỏ |
| 2. Binary search | `f` đơn điệu, tìm `v*`, so `v*−1` | `O(n log M)` | `O(1)` | Không muốn sort, code ngắn |
| 3. Sort + prefix + bisect | `f(v)` trong `O(log n)` | `O(n log n + log M log n)` | `O(n)` | Nhiều truy vấn |
| 4. Sort + tham lam | Chia đều phần còn lại, so `q`/`q+1` | `O(n log n)` | `O(n)` | Bản nộp chuẩn, gọn nhất |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().findBestValue([1, 1, 1], 100)` và `Solution().findBestValue([3, 3, 3], 4)` trả về gì?

**Bài 2 (tính tay):** Chạy tay cách 4 với `arr = [2, 3, 5]`, `target = 9`. Ghi `rem`, `cnt` ở mỗi bước và đáp án.

**Bài 3 (điền TODO):** Hoàn thành `f_fast(a_sorted, prefix, v)` tính `Σ min(a, v)` bằng `bisect` + prefix.

**Bài 4 (tự viết lại):** Viết lại cách 2 (binary search) **không xem code**, đặt tên `my_best_value(arr, target)`.

**Bài 5 (mini challenge):** Biến thể "nâng sàn": thay mọi phần tử **nhỏ hơn** `v` bằng `v`, tìm `v >= 0` để `Σ max(a, v)` gần `target` nhất (hoà chọn nhỏ nhất). Viết `best_floor_value(arr, target)`.

In [29]:
# Bài 3, 4, 5 — khung để tự điền (KHÔNG gọi khi chạy notebook)
def f_fast_todo(a_sorted, prefix, v):
    # TODO: i = số phần tử <= v; trả prefix[i] + v * (len(a_sorted) - i)
    pass

def my_best_value_todo(arr, target):
    # TODO: binary search v* rồi so v*-1 và v*
    pass

def best_floor_value_todo(arr, target):
    # TODO: g(v) = sum(max(a, v)) không giảm; tìm ứng viên quanh ranh giới
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Tổng `[1,1,1]` là 3 < 100. (2) Khi tổng < target, trả `max(arr)`. (3) Với `[3,3,3], 4`: `v = 1` tổng 3 (lệch 1), `v = 2` tổng 6 (lệch 2).

**Bài 2:** (1) Sort xong là `[2,3,5]`. (2) Bước 0: `2·3 = 6 < 9`; bước 1: `rem = 7`, `3·2 = 6 < 7`. (3) Bước 2: `rem = 4`, `cnt = 1`, `5 >= 4` ⇒ `q = 4`.

**Bài 3:** (1) Phần tử `<= v` giữ nguyên. (2) `bisect_right` đếm số phần tử `<= v`. (3) Các phần tử còn lại đóng góp `v` mỗi cái.

**Bài 4:** (1) Xử lý `sum <= target` trước. (2) `first_true` trên `[0, max]`. (3) Hoà ⇒ chọn `v* − 1`.

**Bài 5:** (1) `g(v) = Σ max(a, v)` không giảm; `g(v) = sum(arr)` khi `v <= min(arr)`. (2) Nếu `sum(arr) >= target` thì `v = 0` là tốt nhất (mọi `v` cho `g >= sum >= target`, `v = 0` nhỏ nhất và cho `g` nhỏ nhất). (3) Ngược lại binary search `v*` nhỏ nhất có `g(v) >= target` trong `[0, target]` và so `v*−1`, `v*` — cận trên `target` đủ vì `g(target) >= target`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
assert Solution().findBestValue([1, 1, 1], 100) == 1
assert Solution().findBestValue([3, 3, 3], 4) == 1
# Bài 2
assert trace_greedy([2, 3, 5], 9) == 4

# Bài 3
def f_fast(a_sorted, prefix, v):
    i = bisect.bisect_right(a_sorted, v)
    return prefix[i] + v * (len(a_sorted) - i)

rng = random.Random(8)
for _ in range(300):
    arr = sorted(rng.randint(1, 50) for _ in range(rng.randint(1, 10)))
    pre = [0] + list(accumulate(arr))
    v = rng.randint(0, 60)
    assert f_fast(arr, pre, v) == sum(min(a, v) for a in arr)
print("Bài 1–3: OK")

sorted: [2, 3, 5] | target: 9
i=0 a[i]=2 cnt=3 rem=9 a[i]*cnt=6 -> giữ nguyên 2, rem = 9 - 2 = 7
i=1 a[i]=3 cnt=2 rem=7 a[i]*cnt=6 -> giữ nguyên 3, rem = 7 - 3 = 4
i=2 a[i]=5 cnt=1 rem=4 a[i]*cnt=5 -> dừng: q=4, lệch(q)=0, lệch(q+1)=1 => 4
Bài 1–3: OK


In [31]:
# Bài 4
def my_best_value(arr, target):
    f = lambda v: sum(min(a, v) for a in arr)
    M = max(arr)
    if sum(arr) <= target:
        return M
    lo, hi = 0, M
    while lo < hi:
        mid = (lo + hi) // 2
        if f(mid) >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo - 1 if lo > 0 and target - f(lo - 1) <= f(lo) - target else lo

# Bài 5
def best_floor_value(arr, target):
    g = lambda v: sum(max(a, v) for a in arr)
    if sum(arr) >= target:
        return 0
    lo, hi = 0, target
    while lo < hi:
        mid = (lo + hi) // 2
        if g(mid) >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo - 1 if lo > 0 and target - g(lo - 1) <= g(lo) - target else lo

def best_floor_brute(arr, target):
    best_v, best_d = 0, float("inf")
    for v in range(0, target + 1):
        d = abs(sum(max(a, v) for a in arr) - target)
        if d < best_d:
            best_v, best_d = v, d
    return best_v

rng = random.Random(9)
for _ in range(1000):
    arr = [rng.randint(1, 20) for _ in range(rng.randint(1, 6))]
    target = rng.randint(1, 100)
    assert my_best_value(arr, target) == best_value_brute(arr, target)
    assert best_floor_value(arr, target) == best_floor_brute(arr, target), (arr, target)
print("Bài 4–5: OK — mọi assert đều qua")

Bài 4–5: OK — mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "giá trị gần target nhất" với hàm đơn điệu `f`:**

```python
if f(HI) <= target: return HI_nho_nhat_dat_max   # không bao giờ vượt target
lo, hi = LO, HI
while lo < hi:                                   # v* = nhỏ nhất có f(v) >= target
    mid = (lo + hi) // 2
    if f(mid) >= target: hi = mid
    else:                lo = mid + 1
if lo > LO and target - f(lo - 1) <= f(lo) - target:
    return lo - 1                                # hoà -> nhỏ hơn
return lo
```

**Template tham lam bài 1300:**

```python
a = sorted(arr); rem = target
for i, x in enumerate(a):
    cnt = len(a) - i
    if x * cnt >= rem:
        q = rem // cnt
        return q if rem - q * cnt <= (q + 1) * cnt - rem else q + 1
    rem -= x
return a[-1]
```

**Checklist trước khi submit:**
- [ ] Miền đáp án là `[0, max(arr)]` (có thể là `0`).
- [ ] `sum(arr) <= target` ⇒ trả `max(arr)`.
- [ ] So **hai** ứng viên quanh ranh giới; hoà chọn nhỏ hơn.
- [ ] Không dùng `round()` / số thực.
- [ ] Đúng chữ ký: `findBestValue(self, arr: list[int], target: int) -> int`.